# Brain_Scape — 02 Preprocessing

Run and inspect each preprocessing stage: skull stripping, motion correction, slice timing, denoising, intensity normalization, and atlas registration.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import yaml

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

# Load config
with open('../configs/preprocessing.yaml') as f:
    preproc_config = yaml.safe_load(f)

## 1. Find Input Data

Locate a raw NIfTI scan to preprocess.

In [ ]:
data_dir = Path('../data')
nii_files = sorted(data_dir.rglob('*.nii.gz'))

if nii_files:
    input_path = nii_files[0]
    print(f'Input scan: {input_path}')
    img = nib.load(str(input_path))
    print(f'Shape: {img.shape}, Voxel size: {img.header.get_zooms()}')
else:
    print('No NIfTI files found. Run 01_data_exploration.ipynb and ingest data first.')
    input_path = None

## 2. Skull Stripping

Remove non-brain tissue. FSL BET primary, Otsu fallback.

In [ ]:
if input_path:
    from preprocessing.skull_stripper import SkullStripper

    job_id = 'notebook-demo'
    processed_dir = data_dir / 'processed' / job_id
    processed_dir.mkdir(parents=True, exist_ok=True)

    stripped_path = str(processed_dir / 'stripped.nii.gz')

    stripper = SkullStripper()
    result_path = stripper.strip(str(input_path), stripped_path)
    print(f'Skull-stripped output: {result_path}')

    # Visualize before and after
    stripped_img = nib.load(result_path)
    stripped_data = stripped_img.get_fdata()

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    mid_z = img.shape[2] // 2
    axes[0].imshow(img.get_fdata()[:, :, mid_z].T, cmap='gray', origin='lower')
    axes[0].set_title('Before Skull Stripping')
    axes[0].axis('off')

    axes[1].imshow(stripped_data[:, :, stripped_data.shape[2]//2].T, cmap='gray', origin='lower')
    axes[1].set_title('After Skull Stripping')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    # Brain voxel statistics
    brain_voxels = np.count_nonzero(stripped_data)
    total_voxels = stripped_data.size
    print(f'Brain voxels: {brain_voxels:,} / {total_voxels:,} ({100*brain_voxels/total_voxels:.1f}%)')

## 3. Motion Correction (fMRI only)

For 4D fMRI data, align volumes to a reference. Skip for 3D structural scans.

In [ ]:
if input_path and len(img.shape) == 4:
    from preprocessing.motion_corrector import MotionCorrector

    mc_path = str(processed_dir / 'motion_corrected.nii.gz')
    corrector = MotionCorrector()
    result_path = corrector.correct(stripped_path, mc_path)
    print(f'Motion-corrected output: {result_path}')
else:
    print('Skipping motion correction — input is 3D (structural scan).')
    print('Motion correction applies to 4D fMRI data only.')

## 4. Slice Timing Correction (fMRI only)

Correct for interleaved acquisition. Skip for 3D structural scans.

In [ ]:
if input_path and len(img.shape) == 4:
    from preprocessing.slice_timer import SliceTimer

    st_path = str(processed_dir / 'slice_timed.nii.gz')
    timer = SliceTimer()
    result_path = timer.correct(mc_path, st_path, slice_order='interleaved')
    print(f'Slice-timed output: {result_path}')
else:
    print('Skipping slice timing — input is 3D (structural scan).')

## 5. Denoising

Apply Gaussian smoothing (FWHM=6mm default) to reduce high-frequency noise.

In [ ]:
if input_path:
    from preprocessing.denoiser import Denoiser

    denoised_path = str(processed_dir / 'denoised.nii.gz')
    denoiser = Denoiser()
    result_path = denoiser.denoise(stripped_path, denoised_path)
    print(f'Denoised output: {result_path}')

    denoised_img = nib.load(result_path)
    denoised_data = denoised_img.get_fdata()

    # Compare noise reduction
    orig_nonzero = img.get_fdata()[img.get_fdata() > 0]
    denoised_nonzero = denoised_data[denoised_data > 0]
    noise_reduction = (1 - denoised_nonzero.std() / orig_nonzero.std()) * 100
    print(f'Noise reduction estimate: {noise_reduction:.1f}%')

## 6. Intensity Normalization

Standardize voxel intensity ranges using z-score normalization.

In [ ]:
if input_path:
    from preprocessing.intensity_normalizer import IntensityNormalizer

    normalized_path = str(processed_dir / 'normalized.nii.gz')
    normalizer = IntensityNormalizer()
    result_path = normalizer.normalize(denoised_path, normalized_path)
    print(f'Normalized output: {result_path}')

    norm_img = nib.load(result_path)
    norm_data = norm_img.get_fdata()

    # Verify normalization
    nonzero = norm_data[norm_data != 0]
    print(f'Normalized intensity range: [{nonzero.min():.3f}, {nonzero.max():.3f}]')
    print(f'Mean: {nonzero.mean():.3f}, Std: {nonzero.std():.3f}')

## 7. Atlas Registration

Register the preprocessed scan to MNI152 standard space. This is the most expensive step (5-20 min per scan).

ANTs SyN is the default; FSL FNIRT is the fallback.

In [ ]:
if input_path:
    from preprocessing.atlas_registrar import AtlasRegistrar

    registered_dir = data_dir / 'registered' / job_id
    registered_dir.mkdir(parents=True, exist_ok=True)
    registered_path = str(registered_dir / 'registered.nii.gz')

    registrar = AtlasRegistrar()
    print('Running atlas registration (this may take 5-20 minutes)...')
    result_path, metrics = registrar.register(normalized_path, registered_path)
    print(f'Registered output: {result_path}')
    if metrics:
        print(f'Registration metrics: {metrics}')

## 8. Preprocessing Pipeline Summary

Verify all outputs and compare before/after.

In [ ]:
if input_path:
    stages = [
        ('Original', str(input_path)),
        ('Skull Stripped', stripped_path),
        ('Denoised', denoised_path),
        ('Normalized', normalized_path),
    ]
    if os.path.exists(registered_path):
        stages.append(('Registered', registered_path))

    fig, axes = plt.subplots(1, len(stages), figsize=(5 * len(stages), 5))
    if len(stages) == 1:
        axes = [axes]

    for ax, (name, path) in zip(axes, stages):
        stage_img = nib.load(path)
        stage_data = stage_img.get_fdata()
        mid_z = stage_data.shape[2] // 2
        ax.imshow(stage_data[:, :, mid_z].T, cmap='gray', origin='lower')
        ax.set_title(name)
        ax.axis('off')

    plt.suptitle('Preprocessing Pipeline Stages', fontsize=14)
    plt.tight_layout()
    plt.show()

    print('\nPreprocessing complete! Proceed to 03_reconstruction.ipynb.')